# PrimeVul → GumTree C AST-Diff Validation

## 1. Setup

In [1]:
# Prevents stale src loads
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'src'))

import pandas as pd

from vuln_commit_history import gumtree_runner, primevul_fetch, primevul_manifest, primevul_materialize
from vuln_commit_history.io import read_json, read_jsonl

# Setup
DATA_DIR = Path('..').resolve() / 'data'
DOWNLOADS_ZIP = Path.home() / 'Downloads' / 'PrimeVul_v0.1-20260902T144121Z-1-001.zip'

# Raw inputs
RAW_DIR = DATA_DIR / 'primevul' / 'raw'

# Manifest -- this notebook is the tool-validation exercise (stratified sample + negative
# controls), not the real extraction, hence the validation_* paths throughout.
MANIFEST_PATH = DATA_DIR / 'primevul' / 'manifests' / 'validation_sample_manifest.jsonl'

# Materialized pairs
PAIRS_DIR = DATA_DIR / 'primevul' / 'validation_pairs'

# GumTree
GUMTREE_OUTPUT_DIR = DATA_DIR / 'primevul' / 'validation_gumtree_output'
GUMTREE_JAR = Path('..').resolve() / 'tools' / 'gumtree' / 'lib' / 'gumtree.jar'
JAVA_EXECUTABLE = 'java'

print(DATA_DIR)

C:\Users\kvand\vuln-commit-history\data


In [3]:
paired_paths, source_zip = primevul_fetch.ensure_primevul_raw(RAW_DIR, DOWNLOADS_ZIP)
print('Found PrimeVul raw inputs at', RAW_DIR)

Found PrimeVul raw inputs at C:\Users\kvand\vuln-commit-history\data\primevul\raw


## 2. Candidate pairs

In [4]:
records = primevul_manifest.load_paired_records(paired_paths)
with zipfile.ZipFile(source_zip) as zf:
    candidate_pairs, excluded_pairs = primevul_manifest.build_pairs(records, zf)
print(f'{len(records)} raw records -> {len(candidate_pairs)} pairs kept, {len(excluded_pairs)} excluded')

9408 raw records -> 2602 pairs kept, 17 excluded


## 3. Stratified sample

In [5]:
selected = primevul_manifest.stratified_sample(candidate_pairs)
cwe_count = len({p['primary_cwe'] for p in selected})
print(f'{len(selected)} pairs selected across {cwe_count} CWE categories')

30 pairs selected across 6 CWE categories


## 4. Materialize pairs

In [6]:
materialize_report = primevul_materialize.materialize_pairs(selected, source_zip, PAIRS_DIR)
control_rows = primevul_materialize.materialize_negative_controls(selected, source_zip, PAIRS_DIR)
primevul_materialize.write_manifest(selected, control_rows, MANIFEST_PATH)
print(len(materialize_report['materialized']), 'materialized;', len(materialize_report['failed']), 'failed')
print(len(control_rows), 'negative controls added')

30 materialized; 0 failed
4 negative controls added


## 5. Run GumTree

In [7]:
gumtree_report = gumtree_runner.run_gumtree(
    MANIFEST_PATH, PAIRS_DIR, GUMTREE_OUTPUT_DIR, GUMTREE_JAR, workers=4, java=JAVA_EXECUTABLE
)
print(len(gumtree_report['succeeded']), 'succeeded;', len(gumtree_report['failed']), 'failed')

34 succeeded; 0 failed


## 6. Sanity checks

In [8]:
manifest_rows = read_jsonl(MANIFEST_PATH)
action_totals = []
for row in manifest_rows:
    counts = read_json(GUMTREE_OUTPUT_DIR / row['sample_id'] / 'action_counts.json')
    action_totals.append({'sample_id': row['sample_id'], 'kind': row['kind'], 'total_actions': counts.get('total_actions', 0)})
action_df = pd.DataFrame(action_totals)
action_df

,sample_id,kind,total_actions
0,44554623121680820346289655905713695377__227798...,real,2
1,123413656536164614730473477667931906703__11188...,real,2
2,23594315334891065360640726557315592697__331182...,real,72
3,21610058009013440610924702255406066217__208511...,real,3
4,184974595979519972400668791979192791271__93042...,real,5
5,247944174862272730917153760849014501526__16753...,real,4
6,323749645329160837875108446700628217438__95768...,real,13
7,133576823966685874228817295114021225034__26481...,real,51
8,287438225445565923195507891193203442203__21346...,real,8
9,237185534796488816693477345986057682954__19596...,real,17


In [9]:
# Negative controls should show zero actions
action_df[action_df['kind'].str.startswith('control') & (action_df['total_actions'] > 0)]

,sample_id,kind,total_actions


In [10]:
# Size-bucket vs. action-count sanity (real pairs only)
size_by_sample = {p['sample_id']: p['size_bucket'] for p in selected}
action_df['size_bucket'] = action_df['sample_id'].map(size_by_sample)
action_df.dropna(subset=['size_bucket']).groupby('size_bucket')['total_actions'].describe()

,count,mean,std,min,25%,50%,75%,max
size_bucket,,,,,,,,
large,8.0,121.250000,156.288881,10.0,22.25,52.0,141.25,394.0
medium,9.0,11.333333,6.461424,2.0,8.00,9.0,15.00,21.0
small,13.0,3.076923,2.531848,1.0,1.00,2.0,4.00,8.0
